# Notebook 23 — Comment Rendering Diagnostic

## Purpose

Study 01 displayed a walkover comment as `Walkover<br><br><br>`, raising a bounded database-governance question:

> Is that markup actually stored in Source Version 1 or the accepted Inside Rails database, or was it introduced later during notebook rendering or copy/paste transport?

This notebook investigates that question only.

It does **not** investigate race-time semantics, expose race-time fields, or implement a database extension.

## Boundaries

This notebook must:

- preserve the raw source and accepted-database comment values unchanged;
- inspect the exact stored values and source lineage;
- test whether literal markup or hidden newline characters are present;
- distinguish stored data from rendered or copied presentation;
- avoid authorising any cleaning transformation until the evidence supports one;
- avoid creating a general narrative parser.

This notebook must not:

- alter Source Version 1;
- modify the accepted Inside Rails Version 1 database;
- infer racing meaning from comment prose;
- introduce `comment_plain_text`, HTML stripping, newline stripping or other cleaning without stored-data evidence;
- investigate the separate race-time question.

## Evidence-led workflow

The first bounded question is:

> What markup-like presentation artefacts actually occur in the stored `comment` values?


In [1]:
from pathlib import Path
import sqlite3

import pandas as pd


# Resolve the repository root explicitly rather than relying on the notebook
# having been launched from one particular working directory.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Use the documented immutable Source Version 1 path. This investigation needs
# the exact raw comment values because we are deciding whether a new governed
# presentation transformation is justified.
SOURCE_DB = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "form_2015-present"
    / "form_2015-present"
    / "raceform.db"
)

# Fail closed if the documented source is unavailable rather than silently
# switching to another database or reconstructing the path from memory.
assert SOURCE_DB.exists(), f"Source Version 1 not found: {SOURCE_DB}"


# Open Source Version 1 read-only. This notebook is investigating the evidence;
# it must not create tables, indexes, views or other temporary state in the
# immutable third-party source.
with sqlite3.connect(f"file:{SOURCE_DB}?mode=ro", uri=True) as conn:
    conn.execute("PRAGMA query_only = ON")

    # Count admitted runner rows whose raw comment contains "<".
    # `rowid <> 1` is the governed Source Version 1 admission rule.
    #
    # This gives the size of the markup-like population before we inspect or
    # define any transformation. No assumption is made yet that "<" means HTML.
    markup_row_count = conn.execute(
        """
        SELECT COUNT(*)
        FROM data
        WHERE rowid <> 1
          AND comment LIKE '%<%'
        """
    ).fetchone()[0]

    # Group identical raw comments so repeated presentation patterns become
    # visible without processing all 1.85 million admitted runner rows in
    # Python. The source text remains completely unchanged.
    #
    # We inspect only the most frequent values at this stage because the next
    # question is whether a small, repeatable set of markup artefacts exists.
    common_markup_comments = pd.read_sql_query(
        """
        SELECT
            comment AS raw_comment,
            COUNT(*) AS runner_rows
        FROM data
        WHERE rowid <> 1
          AND comment LIKE '%<%'
        GROUP BY comment
        ORDER BY runner_rows DESC, raw_comment
        LIMIT 50
        """,
        conn,
    )


# Keep the population denominator visible before looking at examples. This
# tells us whether the issue is isolated or material enough to justify a
# reusable study-facing transformation.
print(f"Admitted runner comments containing '<': {markup_row_count:,}")

# Display the exact raw values rather than cleaned versions. We need to inspect
# the evidence before deciding what, if anything, a governed plain-text field
# is permitted to remove.
common_markup_comments

Admitted runner comments containing '<': 0


,raw_comment,runner_rows


### Result: no literal HTML markup was found in the raw comments

The first source-wide check found **zero admitted runner comments containing a
literal `<` character**.

This is negative evidence and changes the proposed database extension.

It establishes that we currently have no evidence that Source Version 1 stores
HTML such as `<br>` inside the `comment` field. We therefore must **not**
introduce a generic HTML-stripping transformation merely because one Study 01
display appeared to contain `<br>` markers.

It does not yet establish where that displayed representation came from. It
could have been introduced after the raw source value was stored, for example
during database retrieval, dataframe preparation or notebook presentation.

The next bounded question is therefore:

> What exact comment values and source lineage are stored in the accepted
> Inside Rails database for the walkover runners?

We will locate the walkovers from their comment content rather than guessing an
exact horse label, course label or raw date representation.

In [2]:
from pathlib import Path

import pandas as pd

from inside_rails.source_sqlite import connect_read_only


# Use the canonical accepted Database v1 path documented for study work.
# This is deliberately the Inside Rails release rather than the third-party
# source file because we are investigating what a study actually receives.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATABASE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "database"
    / "releases"
    / "inside_rails_v1.sqlite3"
)

# Fail closed if the accepted release is unavailable. Study code must not
# silently fall back to the candidate database or immutable third-party source.
assert DATABASE.is_file(), f"Accepted Database v1 not found: {DATABASE}"


with connect_read_only(DATABASE) as connection:
    # Apply the documented accepted-release consumer protections in addition
    # to opening the SQLite file itself in read-only mode.
    connection.execute("PRAGMA query_only = ON")
    connection.execute("PRAGMA foreign_keys = ON")
    connection.execute("PRAGMA trusted_schema = OFF")

    # Restrict the query in SQLite to the small set of runner rows whose raw
    # comments explicitly describe a walkover. This avoids loading the full
    # 1.85-million-row runner population into pandas merely to inspect a few
    # source values.
    #
    # We deliberately do not filter by horse, date or course because the
    # previous failed lookup showed that those raw representations had been
    # assumed rather than established.
    walkover_comments = pd.read_sql_query(
        """
        SELECT
            source_race_occurrence_code,
            runner_participation_code,
            source_record_code,
            source_rowid,
            date AS raw_date,
            course AS raw_course,
            off AS raw_off,
            horse AS raw_horse,
            comment AS raw_comment,
            typeof(comment) AS comment_storage_type,
            length(comment) AS comment_character_length,
            hex(comment) AS comment_utf8_hex
        FROM view_core_runner_participations
        WHERE lower(CAST(comment AS TEXT)) LIKE '%walkover%'
           OR lower(CAST(comment AS TEXT)) LIKE '%walked over%'
        ORDER BY
            CAST(date AS TEXT),
            CAST(course AS TEXT),
            CAST(off AS TEXT),
            source_rowid
        """,
        connection,
    )


# The known walkover investigation should produce at least one matching runner.
# If it does not, stop rather than broadening the search until something fits.
assert not walkover_comments.empty, (
    "No walkover comments were found in the accepted runner-participation view."
)

# `repr()` exposes otherwise invisible characters such as newlines while
# retaining the exact unchanged raw comment beside it. The Boolean diagnostics
# test only presentation characters relevant to the apparent `<br>` issue;
# they do not attempt to interpret the racing narrative.
walkover_comments["comment_repr"] = walkover_comments["raw_comment"].map(repr)
walkover_comments["contains_literal_lt"] = walkover_comments["raw_comment"].map(
    lambda value: "<" in value
)
walkover_comments["contains_line_feed"] = walkover_comments["raw_comment"].map(
    lambda value: "\n" in value
)
walkover_comments["contains_carriage_return"] = walkover_comments["raw_comment"].map(
    lambda value: "\r" in value
)

# Keep source-record and race codes visible because this is a provenance
# investigation rather than reader-facing publication output. These fields let
# us trace any anomalous representation back to one exact accepted source row.
walkover_comments[
    [
        "source_race_occurrence_code",
        "source_record_code",
        "source_rowid",
        "raw_date",
        "raw_course",
        "raw_off",
        "raw_horse",
        "raw_comment",
        "comment_repr",
        "comment_storage_type",
        "comment_character_length",
        "contains_literal_lt",
        "contains_line_feed",
        "contains_carriage_return",
        "comment_utf8_hex",
    ]
]

,source_race_occurrence_code,source_record_code,source_rowid,raw_date,raw_course,raw_off,raw_horse,raw_comment,comment_repr,comment_storage_type,comment_character_length,contains_literal_lt,contains_line_feed,contains_carriage_return,comment_utf8_hex
0,race:77b5dbbbfdee69d4d92a5826:000026626,rec:77b5dbbbfdee69d4d92a5826:data:0000252384,252384,2016-08-14,Southwell,4:05,Generous Chief (IRE),In rear - reminders after 1st and 3rd - headwa...,'In rear - reminders after 1st and 3rd - headw...,text,196,False,False,False,496E2072656172202D2072656D696E6465727320616674...
1,race:77b5dbbbfdee69d4d92a5826:000034708,rec:77b5dbbbfdee69d4d92a5826:data:0000333952,333952,2017-02-22,Ludlow,4:45,The Ould Lad (IRE),Held up - jumped left 10th - mistake and weake...,'Held up - jumped left 10th - mistake and weak...,text,178,False,False,False,48656C64207570202D206A756D706564206C6566742031...
2,race:77b5dbbbfdee69d4d92a5826:000037183,rec:77b5dbbbfdee69d4d92a5826:data:0000357665,357665,2017-04-18,Exeter,5:40,San Satiro (IRE),Walked over,'Walked over',text,11,False,False,False,57616C6B6564206F766572
3,race:77b5dbbbfdee69d4d92a5826:000056130,rec:77b5dbbbfdee69d4d92a5826:data:0000542774,542774,2018-05-23,Kempton (AW),5:55,Vice Marshal (IRE),Behind leaders on inner - ridden 2f out and ev...,'Behind leaders on inner - ridden 2f out and e...,text,143,False,False,False,426568696E64206C656164657273206F6E20696E6E6572...
4,race:77b5dbbbfdee69d4d92a5826:000063363,rec:77b5dbbbfdee69d4d92a5826:data:0000611829,611829,2018-10-04,Warwick,3:15,Desirable Court (IRE),Walked over,'Walked over',text,11,False,False,False,57616C6B6564206F766572
5,race:77b5dbbbfdee69d4d92a5826:000065749,rec:77b5dbbbfdee69d4d92a5826:data:0000636836,636836,2018-11-18,Fontwell,2:40,Cap Horner (FR),Raced in last and jumped slowly at times - hea...,'Raced in last and jumped slowly at times - he...,text,165,False,False,False,526163656420696E206C61737420616E64206A756D7065...
6,race:77b5dbbbfdee69d4d92a5826:000066395,rec:77b5dbbbfdee69d4d92a5826:data:0000643536,643536,2018-12-02,Leicester,3:05,Bailarico (IRE),Walked over,'Walked over',text,11,False,False,False,57616C6B6564206F766572
7,race:77b5dbbbfdee69d4d92a5826:000066741,rec:77b5dbbbfdee69d4d92a5826:data:0000647143,647143,2018-12-12,Kempton (AW),6:15,Dorella (GER),Behind leaders - ridden well over 3f out to ho...,'Behind leaders - ridden well over 3f out to h...,text,103,False,False,False,426568696E64206C656164657273202D2072696464656E...
8,race:77b5dbbbfdee69d4d92a5826:000069358,rec:77b5dbbbfdee69d4d92a5826:data:0000673701,673701,2019-02-23,Lingfield (AW),4:10,Greybychoice (IRE),Walked over,'Walked over',text,11,False,False,False,57616C6B6564206F766572
9,race:77b5dbbbfdee69d4d92a5826:000069743,rec:77b5dbbbfdee69d4d92a5826:data:0000677275,677275,2019-03-03,Sedgefield,5:30,Wazowski (GB),Walked over,'Walked over',text,11,False,False,False,57616C6B6564206F766572


In [3]:
import os
import sys

# Inspect the environment actually seen by this notebook kernel. This tells us
# whether the kernel inherited the PYTHONPATH supplied by the `rails` launcher
# and which Python executable is running it.
print("Python executable:")
print(sys.executable)

print("\nPYTHONPATH:")
print(os.environ.get("PYTHONPATH"))

print("\nRelevant sys.path entries:")
for path in sys.path:
    if "inside-rails-horse-racing" in path:
        print(path)

Python executable:
/home/rob/Documents/inside-rails-horse-racing/.venv/bin/python

PYTHONPATH:
/home/rob/Documents/inside-rails-horse-racing/src

Relevant sys.path entries:
/home/rob/Documents/inside-rails-horse-racing/src
/home/rob/Documents/inside-rails-horse-racing/.venv/lib/python3.12/site-packages


## Conclusion

The apparent `Walkover<br><br><br>` value was **not stored in either Source Version 1 or the accepted Inside Rails database**.

A source-wide check found zero admitted comments containing a literal `<` character. The accepted database row for Queensbury Boy at Hereford on 12 May 2026 stores the comment exactly as `Walkover`, with character length 8 and UTF-8 hexadecimal `57616C6B6F766572`.

The apparent `<br>` markup was introduced after the stored value during rendered-output / copy-paste transport. This was confirmed when copied notebook output also merged material from a separate diagnostic cell into the same pasted representation.

Therefore no comment-cleaning transformation is justified. Existing Notebook 21 comment governance remains unchanged.

**Confidence:** high.

**Manual/external verification:** `not_applicable`.

The separate race-time question is outside this diagnostic.
